# Hindi voice-clone detection: compression test

Scores the baseline detector on test clips that have been through voice-note and phone-call encoding.

Before you start:

1. Run `01_dataset_and_baseline.ipynb` once, including the cell that saves the dataset to Google Drive.
2. Set the runtime to a GPU: Runtime → Change runtime type → T4 GPU.

## Session setup

Run this first in every new session. It connects Google Drive, restores the dataset from the backup, loads AASIST and defines the loading and scoring code.

In [ ]:
import os, json, shutil, subprocess, importlib.util
import numpy as np, pandas as pd, soundfile as sf, torch
from sklearn.metrics import roc_curve
from tqdm.auto import tqdm

DRIVE = "/content/drive/MyDrive/voice-clone-detector"
OUT16 = "/content/data/hindi16k"
AASIST_DIR = "/content/aasist"
RESULTS = f"{DRIVE}/results"

# --- 1. Google Drive ---------------------------------------------------------
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RESULTS, exist_ok=True)

# --- 2. Dataset: restore from the Drive backup if this session does not have it
if not os.path.exists(f"{OUT16}/meta.csv"):
    backup = f"{DRIVE}/hindi16k.zip"
    assert os.path.exists(backup), "hindi16k.zip is not in your Drive folder. Stop here and tell Claude."
    print("restoring the dataset from Google Drive ...")
    shutil.copy(backup, "/content/hindi16k.zip")
    shutil.unpack_archive("/content/hindi16k.zip", OUT16)
meta = pd.read_csv(f"{OUT16}/meta.csv")
missing = [p for p in meta.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} clips are missing, for example {missing[0]}"

# --- 3. AASIST, the baseline detector ---------------------------------------
if not os.path.exists(AASIST_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/clovaai/aasist", AASIST_DIR], check=True)
spec = importlib.util.spec_from_file_location("aasist_model", f"{AASIST_DIR}/models/AASIST.py")
aasist = importlib.util.module_from_spec(spec)
spec.loader.exec_module(aasist)

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = json.load(open(f"{AASIST_DIR}/config/AASIST.conf"))["model_config"]
model = aasist.Model(cfg).to(device)
model.load_state_dict(torch.load(f"{AASIST_DIR}/models/weights/AASIST.pth", map_location=device))
model.eval()

# --- 4. Shared loading and scoring code -------------------------------------
CLIP_LEN = 64600      # about 4 seconds, the length AASIST was trained on
TARGET_DB = -23.0     # every clip is set to this loudness


def load_clip(path):
    a, sr = sf.read(path, dtype="float32")
    assert sr == 16000, f"{path} is {sr} Hz"
    a = a - a.mean()
    a = a * (10 ** (TARGET_DB / 20) / (np.sqrt(np.mean(a ** 2)) + 1e-9))
    if len(a) < CLIP_LEN:
        a = np.tile(a, CLIP_LEN // len(a) + 1)
    return a[:CLIP_LEN]


def score_clips(paths, batch=32, desc="scoring"):
    """Returns one number per clip. Higher means the detector thinks it is real."""
    realness, says_real = [], []
    for i in tqdm(range(0, len(paths), batch), desc=desc):
        x = torch.from_numpy(np.stack([load_clip(p) for p in paths[i:i + batch]])).to(device)
        with torch.no_grad():
            _, out = model(x)
        realness += out[:, 1].cpu().tolist()
        says_real += (out[:, 1] > out[:, 0]).cpu().tolist()
    return np.array(realness), np.array(says_real)


def eer(is_real, score):
    """Equal Error Rate in percent. 0 is perfect, 50 is a coin flip."""
    fpr, tpr, _ = roc_curve(is_real, score)
    i = np.nanargmin(np.abs((1 - tpr) - fpr))
    return 100 * (fpr[i] + 1 - tpr[i]) / 2


def eer_range(is_real, score, n=500, seed=0):
    """Rough 95% range for the EER, from re-drawing the test clips many times."""
    rng, vals = np.random.default_rng(seed), []
    for _ in range(n):
        j = rng.integers(0, len(score), len(score))
        if 0 < is_real[j].sum() < len(j):
            vals.append(eer(is_real[j], score[j]))
    return np.percentile(vals, [2.5, 97.5])


print("\n=== SETUP ===")
print("Running on:", "GPU" if device == "cuda" else "CPU (slow: switch the runtime to T4 GPU and run this again)")
print("Clips:", len(meta), "| test clips:", (meta.split == "test").sum())
print("Results folder:", RESULTS)
print("Files already there:", sorted(os.listdir(RESULTS)))

## Compression test

Makes four degraded copies of every test clip and scores all five versions with AASIST.

| Name | What it imitates |
|---|---|
| `opus_24k` | Voice note, Opus codec at 24 kbps |
| `opus_12k` | Voice note on a poor connection, Opus at 12 kbps |
| `phone_g711` | Landline or internet call: 300 to 3400 Hz band, 8 kHz, G.711 A-law |
| `phone_gsm` | 2G mobile call: 300 to 3400 Hz band, 8 kHz, GSM full-rate |

Most mobile calls today use the AMR codec, which the ffmpeg build on Colab cannot encode. GSM full-rate is the nearest available stand-in.

In [ ]:
import os, subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor

COND_DIR = "/content/data/hindi16k_conditions"
PHONE_BAND = "highpass=f=300,lowpass=f=3400"      # the narrow band a phone line carries

# name: (how ffmpeg encodes it, file type)
CONDITIONS = {
    "opus_24k":   (["-c:a", "libopus", "-b:a", "24k", "-application", "voip"], "ogg"),   # voice note
    "opus_12k":   (["-c:a", "libopus", "-b:a", "12k", "-application", "voip"], "ogg"),   # voice note, poor connection
    "phone_g711": (["-af", PHONE_BAND, "-ar", "8000", "-c:a", "pcm_alaw"], "wav"),       # landline / internet call
    "phone_gsm":  (["-af", PHONE_BAND, "-ar", "8000", "-c:a", "libgsm"], "gsm"),         # 2G mobile call
}

have = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
for name in list(CONDITIONS):
    codec = CONDITIONS[name][0][CONDITIONS[name][0].index("-c:a") + 1]
    if f" {codec} " not in have:
        print(f"skipping {name}: this ffmpeg has no {codec} encoder")
        del CONDITIONS[name]


def degrade(src, dst, args, ext):
    """Encode a clip the way a call or voice note would, then decode it back to 16 kHz WAV."""
    if os.path.exists(dst):
        return
    with tempfile.TemporaryDirectory() as tmp:
        mid = f"{tmp}/x.{ext}"
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", src, "-ac", "1", *args, mid], check=True)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", mid, "-ac", "1", "-ar", "16000",
                        "-c:a", "pcm_s16le", dst + ".tmp.wav"], check=True)
    os.replace(dst + ".tmp.wav", dst)


test = meta[meta.split == "test"].reset_index(drop=True)
test["is_real"] = (test.label == "real").astype(int)

# --- 1. Make a degraded copy of every test clip, for every condition ---------
paths = {"clean": list(test.path)}
for name, (args, ext) in CONDITIONS.items():
    for label in ("fake", "real"):
        os.makedirs(f"{COND_DIR}/{name}/{label}", exist_ok=True)
    paths[name] = [f"{COND_DIR}/{name}/{label}/{os.path.basename(p)}" for p, label in zip(test.path, test.label)]
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as pool:
        jobs = [pool.submit(degrade, s, d, args, ext) for s, d in zip(test.path, paths[name])]
        for j in tqdm(jobs, desc=f"encoding {name}"):
            j.result()

# --- 2. Score every condition with AASIST ------------------------------------
rows, scores = [], []
for name, plist in paths.items():
    realness, says_real = score_clips(plist, desc=f"scoring {name}")
    d = test[["path", "label", "tool", "speaker_id", "is_real"]].assign(
        condition=name, cond_path=plist, realness=realness, says_real=says_real)
    scores.append(d)
    real = d[d.label == "real"]
    row = {"condition": name, "EER all": eer(d.is_real.to_numpy(), d.realness.to_numpy())}
    for t in ("freevc24", "xtts_v2"):
        sub = pd.concat([real, d[d.tool == t]])
        row[f"EER {t}"] = eer(sub.is_real.to_numpy(), sub.realness.to_numpy())
    row["real called fake %"] = 100 * (1 - real.says_real.mean())
    for t in ("freevc24", "xtts_v2"):
        row[f"{t} passed as real %"] = 100 * d[d.tool == t].says_real.mean()
    rows.append(row)

summary = pd.DataFrame(rows).set_index("condition").round(1)
pd.concat(scores).to_csv(f"{RESULTS}/aasist_conditions_hindi_scores.csv", index=False)
summary.to_csv(f"{RESULTS}/aasist_conditions_hindi_summary.csv")

print("\n=== AASIST on Hindi test clips, clean and degraded ===")
print(summary.to_string())
print(f"\n{len(test)} test clips per condition. Saved to {RESULTS}")